# PhyloMInt: Run

This notebook explains how to run PhyloMint, and must be run **AFTER** retrieving BiGG SBML file (see notebook [01_get_sbml_BiGG.ipynb](./01_get_sbml_BiGG.ipynb)), getting objective and targets (see notebook [02_get_objectives.ipynb](./02_get_objectives.ipynb) for both). It compute the run for to get the union of seeds inferred by PhyloMInt (no individual set of seeds given).

> Note:
>
> The PhyloMInt (seed searching, scopes, supplementary data) result files are available: [https://doi.org/10.57745/OS1JND](https://doi.org/10.57745/OS1JND)
>
> After downloadind and unzipping the package, go to 
> - PhyloMInt Seed results + scopes + fluxes + supp timer data to get 10 solutions: analyses/results/PhyloMInt

## **WARNING**
This notebook will run PhyloMInt for e_coli_core from BiGG. No time limit is set for PhyloMInt seed search. After getting solutions, it will run scopes inferred by seed and get computational time. 

In the paper, no time limit is set to get PhyloMInt data. It was run for all 107 networks. 

To avoid a long time running within the notebook, the notebook will copy e_coli_core in a sbml directory on a path that you can change.

## Requirements
Module *networkx*

> Advice:
> 
> Use a conda env called s2lp with python 3.10 for plafrim cluster scripts

In [ ]:
!pip install networkx

## **Slurm-based cluster**: Reproducing paper data
Slurm-based scripts for cluster are available for all networks:
- Launch if needed 
    - [01_job_retrieve_bigg_sbml.sh](../../scripts/plafrim_cluster/01_job_retrieve_bigg_sbml.sh): `sbatch 01_job_retrieve_bigg_sbml.sh`
    - [02_job_get_objective.sh](../../scripts/plafrim_cluster/02_job_get_objective.sh): `sbatch 02_job_get_objective.sh`
    - or copy your local files into you cluster
- Change **_source_** variable by the path of your conda environement with seed2lp installed in files: 
    - [12_1_job_phylomint_seadsearch.sh](../../scripts/plafrim_cluster/12_1_job_phylomint_seadsearch.sh)
    - [12_2_job_phylomint_scope_flux.sh](../../scripts/plafrim_cluster/12_2_job_phylomint_scope_flux.sh)
    - [12_3_job_phylomint_scope_analyse.sh](../../scripts/plafrim_cluster/12_3_job_phylomint_scope_analyse.sh) 
- launch:
    - [12_1_job_phylomint_seadsearch.sh](../../scripts/plafrim_cluster/12_1_job_phylomint_seadsearch.sh)
    - [12_2_job_phylomint_scope_flux.sh](../../scripts/plafrim_cluster/12_2_job_phylomint_scope_flux.sh)
    - [12_3_job_phylomint_scope_analyse.sh](../../scripts/plafrim_cluster/12_3_job_phylomint_scope_analyse.sh)

## **LAUNCH**

### Variable to change (if wanted)

In [1]:
analyse_dir = "../../analyses"
data_dir  = f"{analyse_dir}/data/"
#result_dir=f"{analyse_dir}/results"
result_dir=f"../../results"
temp_dir = "../../tmp/"

### Execute

In [2]:
from os import path, makedirs, listdir
from shutil import copyfile

In [26]:
sbml_dir = f"{data_dir}/bigg/sbml"
#e_coli_dir = f"{data_dir}/bigg/sbml_e_coli_core"
e_coli_dir = f"{result_dir}/sbml/sbml_e_coli_core"
phylomint_result_dir = f"{result_dir}/phylomint"
phylomint_solution_dir=f"{phylomint_result_dir}/seeds_results"
phylomint_flux_dir=f"{phylomint_result_dir}/fluxes"
phylomint_scope_dir=f"{phylomint_result_dir}/scopes"
objective_dir = f"{data_dir}/objective"
target_dir = f"{data_dir}/target"

In [4]:
if not path.isdir(e_coli_dir):
    makedirs(e_coli_dir)
    copyfile(path.join(sbml_dir, "e_coli_core.xml"), path.join(e_coli_dir, "e_coli_core.xml"))

#### Function

This function will execute the original seed search source  code extracted from PhyloMInt and the corrected code including reversible reaction into the graph for e_coli_core

In [4]:
if not path.isdir(phylomint_result_dir):
    makedirs(phylomint_result_dir)

sbml_file=path.join(e_coli_dir,"e_coli_core.xml")
objective_file=path.join(objective_dir,"e_coli_core_target.txt")
target_file=path.join(target_dir,"e_coli_core_targets.txt")

In [7]:
file = "../../scripts/12_1_phylomint_seedsearch.py"

In [12]:
!python {file} {sbml_file} {objective_file} {phylomint_result_dir}

### **Scopes and fluxes**

This notebook compute scope for e_coli_core:
- PhyloMInt results files 

#### Function

In [25]:
def run_scope_flux(solution_dir:str, flux_dir:str, scope_dir:str):
    for filename in listdir(e_coli_dir):
        species = f'{path.splitext(path.basename(filename))[0]}'
        sbml_path = path.join(e_coli_dir,filename)

        solution_path=solution_dir
        for filename_solution in listdir(solution_path):
            if species in filename_solution and "results.json" in filename_solution:
                flux_path = path.join(f"{flux_dir}",species)
                scope_path = path.join(f"{scope_dir}",species)
                file_solution_path=path.join(solution_path,filename_solution)

                command_flux = f"flux {sbml_path} {file_solution_path} {flux_path}"
                command_scope = f"scope {sbml_path} {file_solution_path} {scope_path}"
                !seed2lp {command_flux}
                !seed2lp {command_scope}

#### Run

In [28]:
run_scope_flux(phylomint_solution_dir, phylomint_flux_dir, phylomint_scope_dir)

           
                       _   ___    _   
  ___   ___   ___   __| | |_  \  | | _ __  
 / __| / _ \ / _ \ / _` |   ) |  | || '_ \ 
 \__ \|  __/|  __/| (_| |  / /_  | || |_) |
 |___/ \___| \___| \__,_| |____| |_|| .__/    
                                    |_|         
      
Network name: e_coli_core

Finding objective ...
____________________________________________


############################################
############################################
                 CHECK FLUX
############################################
############################################

---------------- FLUX INIT -----------------

{'BIOMASS_Ecoli_core_w_GAM': 0.8739215069684295}


--------------- MEDIUM INIT ----------------

EX_co2_e -1000.0 1000.0
EX_glc__D_e -10.0 1000.0
EX_h_e -1000.0 1000.0
EX_h2o_e -1000.0 1000.0
EX_nh4_e -1000.0 1000.0
EX_o2_e -1000.0 1000.0
EX_pi_e -1000.0 1000.0


---------- STOP IMPORT FLUX -------------

{'BIOMASS_Ecoli_core_w_GAM': 0.0}



___________________

### **Scope Analyses**

#### Function

In [29]:
def run_scope_analyses(sbml_dir:str, scope_dir:str, obj_dir:str,
                       lev1:list, lev2:list, lev3:list, lev4:list,
                       is_cobrapy:bool=False):
    file = "../../scripts/10_1_scope_analyse.py"
    for filename in listdir(sbml_dir):
        species = f'{path.splitext(path.basename(filename))[0]}'
        sbml_path = path.join(sbml_dir, filename)
        objective_path = path.join(obj_dir,f"{species}_target.txt")
        dir = list(map(lambda x: x.split('.')[0], listdir(scope_dir)))
        if species in dir:
            path_seed=path.join(scope_dir, species, 'sbml')
            path_scope=path.join(scope_dir, species, 'scope')
            for l1 in lev1:
                for l2 in lev2:
                    for l3 in lev3:
                        for l4 in lev4:
                            modes_info = path.join(l1, l2, l3, l4)
                            full_path_seed = path.join(path_seed, modes_info)
                            full_path_scope = path.join(path_scope, modes_info)
                            if is_cobrapy:
                                modes_info = "cobrapy"
                            if path.isdir(full_path_scope):
                                command=f"{species} {sbml_path} {full_path_scope} {full_path_seed} {objective_path} {modes_info}"
                                !python {file} {command}

#### Run

In [30]:
list_dir_lev1=["phylomint"]
list_dir_lev2=["phylomint"]
list_dir_lev3=["minimize"]
list_dir_lev4=["accu"]

In [31]:
run_scope_analyses(e_coli_dir, phylomint_scope_dir, objective_dir, list_dir_lev1, list_dir_lev2, list_dir_lev3, list_dir_lev4)

           
                       _   ___    _   
  ___   ___   ___   __| | |_  \  | | _ __  
 / __| / _ \ / _ \ / _` |   ) |  | || '_ \ 
 \__ \|  __/|  __/| (_| |  / /_  | || |_) |
 |___/ \___| \___| \__,_| |____| |_|| .__/    
                                    |_|         
      
